#Ingestion de un archivo JSON

#1. leemos el archivo JSON usando "DataFrameReader" de Spark

In [0]:
%run "../includes/librerias"

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
#Definimos parametros
v_archivo = "country"
v_esquema = "movie_silver"
v_tabla = "countries"
v_partition = "file_date"
dbutils.widgets.text("p_esquema", v_esquema)
dbutils.widgets.text("p_tabla", v_tabla)


In [0]:
# Definimos el schema

country_schema = "countryId INT, countryIsoCode STRING, countryName STRING"


In [0]:
country_df = spark.read\
    .schema(country_schema)\
    .json(f"{bronze_folder_path}/{v_file_date}/{v_archivo}.json")

display(country_df)

##Paso 2 - Seleccionar y / o eliminar las columnas que se requieren

In [0]:
country_dropped_df = country_df.drop(col("countryIsoCode"))


##Paso 3 - Cambiar de nombre de las columnas y anadir columnas


In [0]:
from pyspark.sql.functions import current_timestamp, lit

countries_final_df = country_dropped_df\
    .withColumnRenamed("countryId", "country_id")\
    .withColumnRenamed("countryName", "country_name")

countries_final_df = add_ingestion_date(countries_final_df)
countries_final_df = add_env(countries_final_df)  
countries_final_df = add_file_date(countries_final_df)

display(countries_final_df)


##Paso 4 - Guardar datos en datalake en formato parket

In [0]:

countries_final_df.write.mode("overwrite").format("delta").saveAsTable(f"{v_esquema}.{v_tabla}")
print(f"Se insertaron {countries_final_df.count()} registros en la tabla {v_esquema}.{v_tabla}")


In [0]:
dbutils.notebook.exit("El notebook 04. Ingestion File country.json, termino correctamente")